# Propose data quality rules with an LLM

For each table in `source_tables`: profile it, send the **profile** (not the data) to the model, check every proposed rule with Spark, and save the results to the review tables. Nothing here touches the source tables or the enforced rules.

The logic lives in `dq_core.py`, which must be in the same folder as this notebook. Review proposals with `03_review_and_approve.sql`.

In [0]:
%pip install -U -q langchain-openai

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("source_tables", "", "Tables to profile (comma-separated, catalog.schema.table)")
dbutils.widgets.text("tag", "", "Rule tag (optional)")
dbutils.widgets.text("dq_schema", "workspace.data_quality", "Data quality schema")
dbutils.widgets.dropdown("llm_provider", "databricks", ["databricks", "openai"], "Model provider")
dbutils.widgets.text("model", "system.ai.gpt-oss-120b", "Model: system.ai.<name> (databricks) or OpenAI model name (openai)")
dbutils.widgets.text("secret_scope", "llm", "Secret scope (openai only)")
dbutils.widgets.text("secret_key", "openai_api_key", "Secret key (openai only)")
dbutils.widgets.text("profile_rows", "100000", "Rows to sample for profiling")
dbutils.widgets.dropdown("send_values", "true", ["true", "false"], "Send value counts / examples to the model")

SOURCE_TABLES = ["workspace.default.demand", "workspace.default.prices"]
TAG = dbutils.widgets.get("tag").strip() or None
DQ_SCHEMA = dbutils.widgets.get("dq_schema").strip()
LLM_PROVIDER = dbutils.widgets.get("llm_provider")
MODEL = dbutils.widgets.get("model").strip()
PROFILE_ROWS = int(dbutils.widgets.get("profile_rows"))
SEND_VALUES = dbutils.widgets.get("send_values") == "true"

assert SOURCE_TABLES, "Set source_tables, e.g. workspace.default.demand,workspace.default.prices"
for t in SOURCE_TABLES:
    assert t.count(".") == 2, f"{t!r} must be a full name: catalog.schema.table"

In [0]:
import os
import sys
import uuid

from pyspark.sql import functions as F

# dq_core.py has to sit in the same workspace folder as this notebook, as a
# file (file icon), not a notebook. The notebook's folder is the working directory.
sys.path.insert(0, os.getcwd())
assert os.path.exists("dq_core.py"), (
    f"dq_core.py not found in {os.getcwd()}. Import it into the same folder as this notebook. "
    "If it shows a notebook icon, delete it and import it again as a file.")
import dq_core

from langchain_openai import ChatOpenAI

if LLM_PROVIDER == "databricks":
    # A Databricks-hosted model through Unity Gateway: no API key, and the profile
    # stays in Databricks. Model names are system.ai.<name>; see Serving -> Unity Gateway.
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()  # picks up the notebook's own login
    token = w.config.authenticate()["Authorization"].split(" ", 1)[1]
    llm = ChatOpenAI(model=MODEL, api_key=token,
                     base_url=f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1")
else:
    # OpenAI directly. Needs outbound internet to api.openai.com from serverless compute.
    llm = ChatOpenAI(model=MODEL, api_key=dbutils.secrets.get(
        scope=dbutils.widgets.get("secret_scope"), key=dbutils.widgets.get("secret_key")))

run_id = str(uuid.uuid4())
me = spark.sql("SELECT current_user()").first()[0]
print(f"run_id = {run_id}")

for table in SOURCE_TABLES:
    try:
        rule_recs, mapping_recs = dq_core.propose_for_table(
            spark, llm, table, dq_schema=DQ_SCHEMA, run_id=run_id, user=me, model_name=MODEL,
            tag=TAG, profile_rows=PROFILE_ROWS, send_values=SEND_VALUES)
        print(f"{table}: {sum(r['status'] == 'proposed' for r in rule_recs)} rules for review "
              f"({sum(r['status'] == 'invalid' for r in rule_recs)} invalid), "
              f"{sum(m['status'] == 'proposed' for m in mapping_recs)} mappings for review "
              f"({sum(m['status'] == 'invalid' for m in mapping_recs)} invalid)")
    except Exception as e:  # one bad table shouldn't stop the rest
        print(f"{table}: FAILED - {str(e).splitlines()[0][:300]}")

In [0]:
display(
    spark.read.table(f"{DQ_SCHEMA}.rule_proposals").where(F.col("run_id") == run_id)
    .select("target_table", "status", "rule_name", "constraint_expr", "action",
            F.round(F.col("fail_rate") * 100, 2).alias("fail_pct"), "rationale", "validation_error")
    .orderBy("target_table", "status", F.desc("fail_pct"))
)

In [0]:
display(
    spark.read.table(f"{DQ_SCHEMA}.mapping_proposals").where(F.col("run_id") == run_id)
    .select("target_table", "status", "column_name", "raw_value", "clean_value", "rows_in_sample",
            "rationale", "validation_error")
    .orderBy("target_table", "status", "column_name", F.desc("rows_in_sample"))
)